In [1]:
import os
import sys
import json
import time
import argparse
import random
import glob
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from transformers import AutoTokenizer, AutoConfig
from safetensors.torch import load_file
import pyfaidx
import dotenv
from pathlib import Path
import re

# 导入自定义模块（需确保 src 目录在 Python 路径中）
from src.dataset import MultiTrackDataset, load_fasta_sequence
from src.viewer import DatasetViewer, ResultsViewer
from src.model import GenOmics, load_finetuned_model

# 设置中文字体（根据系统环境调整）
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
class MultiTrackPredictor:
    def __init__(self, fasta_path: str, base_model_path: str, sft_ckpt_path: str,
                 tokenizer_path: str, use_flash_attn: bool,
                 index_stat_path: str = None, index_stat: dict = None,
                 **kwargs):
        self.fasta = pyfaidx.Fasta(fasta_path)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        
        # 支持直接传 dict 或从 JSON 文件读取
        if index_stat is not None:
            stat_data = index_stat
        elif index_stat_path is not None:
            stat_data = json.load(open(index_stat_path, "r"))
        else:
            raise ValueError("必须提供 index_stat（dict）或 index_stat_path（文件路径）")
        
        self.model = load_finetuned_model(
            model_class = GenOmics,
            model_path = base_model_path,
            ckpt_path = sft_ckpt_path,
            use_flash_attn = use_flash_attn,
            device = "cuda:0",
            model_init_kwargs={"index_stat": stat_data, **kwargs}
        )
        print("✔️ Model loaded successfully")
        print(self.model)
        self.model.eval()

    def predict(self, chrom: str, start: int, end: int, biosample_names: list = None) -> dict:
            predict_sequence = load_fasta_sequence(self.fasta, chrom, start, end)
            inputs = self.tokenizer(
                predict_sequence,
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=32768,
                add_special_tokens=False
            ).to("cuda")
            with torch.no_grad():
                start_time = time.time()
                result = self.model.predict(inputs['input_ids'],
                                            biosample_names=biosample_names)
                time_taken = time.time() - start_time
                torch.cuda.empty_cache()
            print(f"Inference time: {time_taken:.2f} s.")
            return {
                'sequence': predict_sequence,
                'position': (chrom, start, end),
                'values': result
            }
    def predict2(self, chrom: str, start: int, end: int, seq: str, biosample_names: list = None) -> dict:
        sequence = pyfaidx.Fasta(seq)
        predict_sequence = str(sequence[0][:])
        inputs = self.tokenizer(
            predict_sequence,
            return_tensors="pt",
            padding=False,
            truncation=True,
            max_length=32768,
            add_special_tokens=False
        ).to("cuda")
        with torch.no_grad():
            start_time = time.time()
            result = self.model.predict(inputs['input_ids'],
                                        biosample_names = biosample_names)
            time_taken = time.time() - start_time
            torch.cuda.empty_cache()
        print(f"Inference time: {time_taken:.2f} s.")
        return {
            'sequence': predict_sequence,
            'position': (chrom, start, end),
            'values': result
        }

In [8]:
class Args:
    pass

args = Args()
args.output_dir = './mutant'
args.smoothing_sigma = 10
args.biosample_names = "NIP_CSQ"

base_model_dir = "/mnt/rice/default/Workspace/xz/hf/rice_1B_stage2_8k_hf"
sft_ckpt_path = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/output/202604020731/checkpoint-23540/model.safetensors"
fasta_path = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/osa1_r7.asm.ch.fa"
ANNOTATION_PATH = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/mutant_output_260525/modified_osa1_r7.all_models.gff3"
test_data_dir = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/riceNavi_output"
riceNavi_csv = os.path.join(test_data_dir, "riceNavi.txt")
index_stat_path = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/index_stat3.json"

chrom="Chr9"
window_start=20716773
window_end=20749541

In [4]:
# 读取 riceNavi.txt
if not os.path.exists(riceNavi_csv):
    print(f"Error: riceNavi.txt not found at {riceNavi_csv}")
    sys.exit(1)
df_riceNavi = pd.read_csv(riceNavi_csv, sep="\t")

# 设置随机种子
seed = 42
random.seed(seed)
np.random.seed(seed)
os.environ["PYTHONHASHSEED"] = str(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
print(f"✅ Random seed set to {seed}")

# 初始化预测器
predictor = MultiTrackPredictor(
    fasta_path=fasta_path,
    base_model_path=base_model_dir,
    sft_ckpt_path=sft_ckpt_path,
    tokenizer_path=base_model_dir,
    use_flash_attn=True,
    index_stat_path=index_stat_path,
    # index_stat=index_stat,
    proj_dim=1024,
    num_downsamples=4,
    bottleneck_dim=1536
)

✅ Random seed set to 42


2026-08-12 11:19:24,314 - INFO - ⚠️ 使用 Flash Attention 2 需要 torch.float16 或 torch.bfloat16，已自动设置为 torch.bfloat16
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in MixtralModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", dtype=torch.float16)`


✔️ Model loaded successfully
GenOmics(
  (base): MixtralModel(
    (embed_tokens): Embedding(128, 1024, padding_idx=14)
    (layers): ModuleList(
      (0-11): 12 x MixtralDecoderLayer(
        (self_attn): MixtralAttention(
          (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1024, out_features=512, bias=False)
          (v_proj): Linear(in_features=1024, out_features=512, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (block_sparse_moe): MixtralSparseMoeBlock(
          (gate): Linear(in_features=1024, out_features=8, bias=False)
          (experts): ModuleList(
            (0-7): 8 x MixtralBlockSparseTop2MLP(
              (w1): Linear(in_features=1024, out_features=4096, bias=False)
              (w2): Linear(in_features=4096, out_features=1024, bias=False)
              (w3): Linear(in_features=1024, out_features=4096, bias=False)
              (act_fn): SiLUAc

In [9]:
# 预测参考序列
print("\n--- Predicting reference sequence ---")
ref_predict = predictor.predict(chrom=chrom, start=window_start, end=window_end,
                                biosample_names=args.biosample_names)


--- Predicting reference sequence ---
Inference time: 1.80 s.


In [10]:
ref_predict

{'sequence': '...',
 'position': ('Chr9', 20716773, 20749541),
 'values': {'total_RNA-seq_+': {'NIP_CSQ': tensor([[[1.4453],
            [1.7109],
            [1.3438],
            ...,
            [0.0167],
            [0.0483],
            [0.0801]]], device='cuda:0', dtype=torch.bfloat16)}}}

In [7]:
ref_predict['values'].keys()

dict_keys(['total_RNA-seq_+'])